# 05 · MAP / LPP Compliance Logic  *(Task 4, part 2)*

**Goal:** judge every price observation against the manufacturer's pricing policy and store the verdict in `PRICE_MONITORING_TABLE`
(one row per observation, with the reason and all numbers used).

## The policy in plain words
* **LPP** (Lowest Possible Price) is a *hard floor*. Advertising below it is **always a violation**, even during a promotion.
* **MAP** (Minimum Advertised Price) is a *soft floor*. Advertising below it is a violation **unless** an approved promotion that quarter allows that lower price.
* A promotion lowers the allowed price to the **promotional price**: `MAP × (1 − % off)` for percent offers, or `MAP − dollar-off converted to INR` for dollar offers.

## Decision order (first rule that applies wins)
1. Missing price / MAP / LPP / unknown currency → **cannot be judged** (reported separately, not counted as violation)
2. Advertised price below **LPP** → **violation**
3. Advertised price at or above the **promotional price** → compliant (promotion covers it)
4. Advertised price below **MAP** → **violation**
5. Otherwise → compliant

## Currencies
Sellers advertise in USD, GBP, AED, CAD or INR. Everything is converted to **INR** (the currency of MAP/LPP) with exchange rates fetched live and **saved per day**
(so a past run can be reproduced). If the live service is unreachable, the newest saved rates are used, and as a last resort built-in rates (the notebook says which).

## Settings that change results — read before running
* **`QUARTER_MODE`**: the offer grid is defined on a *fiscal* year (Nov–Jan = Q1, Feb–Apr = Q2, May–Jul = Q3, Aug–Oct = Q4). The earlier version used calendar quarters, which put 21–31 Dec in Q4. The default here is `"fiscal"` to match the offer grid; switch to `"calendar"` to reproduce the old behaviour.
* **Percent promotions** now work as described above. In the earlier version a missing dollar value was stored as `0`, which turned every percent promotion into "MAP minus $0" — so a legitimately promoted price below MAP was wrongly flagged as a violation.

**Needs:** notebooks 00, 02 and 04 run.

**What this cell does:** Connects Google Drive, finds your project folder and loads the shared helper module `map_common`.

**Purpose:** Every notebook starts here so they all use the same folders, database settings and helpers.

In [ ]:
# ---- Where the project lives on Google Drive (change only if you used a different folder name) ----
PROJECT_DIR = "/content/drive/MyDrive/MAP_Compliance_Project"

import os, sys, importlib
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Colab - using the folder path as given.")

if not os.path.exists(os.path.join(PROJECT_DIR, "map_common.py")):
    raise FileNotFoundError("map_common.py not found. Run notebook 00_Project_Setup first "
                            "(and check PROJECT_DIR above matches your Drive folder).")
sys.path.insert(0, PROJECT_DIR)
import map_common as mc
importlib.reload(mc)
mc.ensure_dirs()
print("Project folder:", mc.PROJECT_DIR)

**What this cell does:** Starts MySQL (restoring the Drive backup in a new Colab session) and connects.

**Purpose:** Opens the database connection used to read the master data and observations and to write the results.

In [ ]:
import numpy as np
import pandas as pd
from sqlalchemy import text
engine = mc.connect_db()

**What this cell does:** Settings: quarter definition, base currency, which currency dollar promotions are quoted in, and the mapping from region name to currency.

**Purpose:** Business rules that can change are gathered here. Region matching ignores upper/lower case, so 'India', 'INDIA' and 'india' are all the same.

In [ ]:
QUARTER_MODE = "fiscal"          # "fiscal" (Nov-Jan = Q1 ...) matches the offer grid | "calendar" (Jan-Mar = Q1 ...)
BASE_CURRENCY = "INR"            # MAP and LPP are stored in this currency
DOLLAR_PROMO_CURRENCY = "USD"    # '$10 off' means 10 US dollars
FORCE_FX_REFRESH = False         # True = ignore today's saved rates and fetch again
REGION_TO_CURRENCY = {"USA": "USD", "US": "USD", "UK": "GBP", "GB": "GBP", "UAE": "AED",
                      "CAN": "CAD", "CANADA": "CAD", "IN": "INR", "IND": "INR", "INDIA": "INR"}
assert QUARTER_MODE in ("fiscal", "calendar")
print("Quarter mode:", QUARTER_MODE)

**What this cell does:** Gets exchange rates (units of each currency per 1 INR) from the shared helper, which saves a snapshot per day and falls back to older/built-in rates if the internet service is down. Prints the rates and where they came from.

**Purpose:** Converting prices needs a rate for every region. Showing the source makes it clear whether results used live, saved or fallback rates.

In [ ]:
fx_rates, fx_source = mc.get_fx_rates(BASE_CURRENCY, FORCE_FX_REFRESH)
print("FX source:", fx_source)
print({k: round(fx_rates[k], 6) for k in sorted(set(REGION_TO_CURRENCY.values())) if k in fx_rates})
missing_ccy = sorted(set(REGION_TO_CURRENCY.values()) - set(fx_rates))
if missing_ccy:
    print("WARNING: no exchange rate for", missing_ccy, "- rows in those regions cannot be judged")
if "BUILT-IN" in fx_source or "OLDER" in fx_source:
    print("NOTE: not today's live rates - results are approximate. Re-run later with internet access for exact rates.")

**What this cell does:** Reads the observations and the master tables (category, price list with MAP/LPP, seller mapping, promotions) from MySQL.

**Purpose:** Pulls together everything needed for the judgement. All prices are converted from the database's exact decimal type to ordinary numbers for calculation.

In [ ]:
with engine.connect() as conn:
    obs       = pd.read_sql(text("SELECT * FROM SELLER_PRICE_OBSERVATIONS"), conn)
    catmap    = pd.read_sql(text("SELECT SKU, CATEGORY, SUBCATEGORY, PL FROM CATEGORYMAPPING_TABLE"), conn)
    pricelist = pd.read_sql(text("SELECT SKU, `MAP` AS MAP_Price, LPP FROM PRICELIST_TABLE"), conn)
    sellermap = pd.read_sql(text("SELECT SSELLERS_NAME AS Seller_name, HOMOLOGATED_SELLERS AS homologated_name FROM SELLERMAPPING_TABLE"), conn)
    promo     = pd.read_sql(text("SELECT SKU, `QUARTER` AS Season, OFFER_PCT, DOLLAR FROM PROMOTION_TABLE"), conn)
if obs.empty:
    raise RuntimeError("SELLER_PRICE_OBSERVATIONS is empty. Run notebook 04 first.")
for frame, cols in [(obs, ["adv_price"]), (pricelist, ["MAP_Price", "LPP"]), (promo, ["OFFER_PCT", "DOLLAR"])]:
    for c in cols:
        frame[c] = pd.to_numeric(frame[c], errors="coerce")
obs["file_date"] = pd.to_datetime(obs["file_date"])
sellermap = sellermap.drop_duplicates("Seller_name", keep="first")   # one recognised name per spelling
print(f"Observations {len(obs):,} | category rows {len(catmap):,} | price list {len(pricelist):,} | "
      f"seller spellings {len(sellermap):,} | promotions {len(promo):,}")

**What this cell does:** Labels each observation with its quarter (Q1-Q4) using the chosen quarter definition and shows how many observations fall in each.

**Purpose:** Promotions apply per quarter, so the observation date decides which promotion (if any) is in force.

In [ ]:
month = obs["file_date"].dt.month
if QUARTER_MODE == "fiscal":
    q = ((month - 11) % 12) // 3 + 1          # Nov,Dec,Jan -> 1 ; Feb-Apr -> 2 ; May-Jul -> 3 ; Aug-Oct -> 4
else:
    q = obs["file_date"].dt.quarter
obs["Season"] = "Q" + q.astype(str)
print(obs.groupby("Season")["file_date"].agg(["count", "min", "max"]).to_string())

**What this cell does:** Joins the observations to category, MAP/LPP, recognised seller name and the promotion for that SKU and quarter. Checks that the joins did not add or lose any rows and reports how many rows lack a mapping.

**Purpose:** Builds the one wide table the rules run on. The row-count check protects against a duplicate key silently multiplying observations.

In [ ]:
n0 = len(obs)
pm = obs.merge(catmap, on="SKU", how="left")
pm = pm.merge(pricelist, on="SKU", how="left")
pm = pm.merge(sellermap, on="Seller_name", how="left")
pm = pm.merge(promo, on=["SKU", "Season"], how="left")
assert len(pm) == n0, f"Joins changed the row count ({n0} -> {len(pm)}): a master table has duplicate keys."
print("Rows:", f"{len(pm):,}", "(unchanged by joins)")
print("Rows without category mapping :", int(pm["CATEGORY"].isna().sum()))
print("Rows without a price list entry:", int(pm["MAP_Price"].isna().sum()))
print("Rows without LPP               :", int(pm["LPP"].isna().sum()))
print("Rows with seller not in mapping:", int(pm["homologated_name"].isna().sum()), "(raw seller name is used for strikes)")
print("Rows with an active promotion  :", int((pm["OFFER_PCT"].fillna(0).gt(0) | pm["DOLLAR"].fillna(0).gt(0)).sum()))

**What this cell does:** Defines `calc_promo()` - the promotional price: percent offer → `MAP × (1 − pct/100)`; dollar offer → `MAP − dollar ÷ USD-per-INR`; no price offer → empty - and tests it on hand-worked examples. Then applies it to all rows.

**Purpose:** This is the corrected promotion logic. The hand examples (10% off 1000 = 900; $10 at 0.01 USD per INR = 1000 INR off) prove it before it touches real data.

In [ ]:
def calc_promo(map_price, pct, dollar, usd_per_base):
    """Returns (promotional_price, dollar_discount_in_base_currency). Empty (NaN) when no price-based offer applies.
    Dollar offers take precedence over percent offers if a row somehow has both."""
    has_dol = dollar.fillna(0) > 0
    has_pct = (pct.fillna(0) > 0) & ~has_dol
    dol_base = (dollar / usd_per_base).round(2).where(has_dol)
    price = pd.Series(np.nan, index=map_price.index)
    price = price.mask(has_dol, map_price - dol_base)
    price = price.mask(has_pct, map_price * (1 - pct / 100))
    return price.clip(lower=0).round(2), dol_base

# --- hand-worked tests: 10% off 1000 = 900 | $10 at 0.01 USD per INR = 1000 INR off | no offer = empty
t = pd.DataFrame({"map": [1000.0, 5000.0, 1000.0, 1000.0], "pct": [10.0, np.nan, np.nan, np.nan], "dol": [np.nan, 10.0, 0.0, np.nan]})
tp, td = calc_promo(t["map"], t["pct"], t["dol"], 0.01)
assert tp.iloc[0] == 900.0 and tp.iloc[1] == 4000.0 and np.isnan(tp.iloc[2]) and np.isnan(tp.iloc[3]), tp.tolist()
assert td.iloc[1] == 1000.0
print("calc_promo hand tests passed")

usd = fx_rates[DOLLAR_PROMO_CURRENCY]
pm["Promotional_price"], pm["promo_dollar_base"] = calc_promo(pm["MAP_Price"], pm["OFFER_PCT"], pm["DOLLAR"], usd)
pm["fx_rate_used"] = np.where(pm["promo_dollar_base"].notna(), usd, np.nan)
print("Rows with a promotional price:", int(pm["Promotional_price"].notna().sum()),
      "| percent:", int((pm["OFFER_PCT"].fillna(0).gt(0) & pm["promo_dollar_base"].isna()).sum()),
      "| dollar:", int(pm["promo_dollar_base"].notna().sum()))

**What this cell does:** Converts each advertised price from the seller's local currency to INR: finds the currency from the region, divides by the rate (units per 1 INR), and rounds to 2 decimals. Lists any region without a currency.

**Purpose:** MAP and LPP are in INR, so prices must be in INR to be comparable. A region with no currency mapping is surfaced rather than guessed.

In [ ]:
pm["currency_local"] = pm["region"].astype(str).str.strip().str.upper().map(REGION_TO_CURRENCY)
unknown_regions = sorted(pm.loc[pm["currency_local"].isna(), "region"].dropna().unique())
if unknown_regions:
    print("WARNING: no currency for region(s):", unknown_regions, "- add them to REGION_TO_CURRENCY (these rows cannot be judged)")
rate = pm["currency_local"].map(fx_rates)
pm["Advertised_price_base"] = (pm["adv_price"] / rate.where(rate > 0)).round(2)
print("Rows converted:", int(pm["Advertised_price_base"].notna().sum()), "of", len(pm))
pm.groupby("currency_local", dropna=False)["Advertised_price_base"].describe()[["count", "min", "50%", "max"]].round(0)

**What this cell does:** Defines `classify()` - the decision order above, applied to all rows at once - and tests it on seven hand-worked cases (below LPP, promotion not enough, promotion covers, below MAP, compliant, missing data, and a promotion that must NOT excuse an LPP breach).

**Purpose:** This is the heart of the compliance logic. The hand cases act as a unit test, so any later change to the rules that breaks the policy is caught immediately.

In [ ]:
R_UNCLASS = "Cannot classify -- missing price data or unmapped currency"
R_LPP     = "Below LPP (hard floor) -- violation regardless of any promotion"
R_MAP     = "Below MAP with no active/sufficient promotion to authorize it"

def classify(adv, map_p, lpp, promo):
    unclass   = adv.isna() | map_p.isna() | lpp.isna()
    below_lpp = adv < lpp
    covered   = promo.notna() & (adv >= promo)
    below_map = adv < map_p
    reason = np.select([unclass, below_lpp, covered, below_map], [R_UNCLASS, R_LPP, "", R_MAP], default="")
    violation = ~unclass & (below_lpp | (~covered & below_map))
    return violation.to_numpy(), pd.Series(reason, index=adv.index).replace("", None).to_numpy(dtype=object)

nan = np.nan
cases = pd.DataFrame([  # adv, MAP, LPP, promo, expected violation, expected reason
    (700,  1000, 800, nan, True,  R_LPP),      # below LPP
    (850,  1000, 800, 900, True,  R_MAP),      # promo exists but price is under even the promo price
    (950,  1000, 800, 900, False, None),       # promotion covers a below-MAP price
    (950,  1000, 800, nan, True,  R_MAP),      # below MAP, no promotion
    (1000, 1000, 800, nan, False, None),       # at MAP
    (nan,  1000, 800, nan, False, R_UNCLASS),  # missing price
    (700,  1000, 800, 600, True,  R_LPP),      # promotion never excuses an LPP breach
], columns=["adv", "map", "lpp", "promo", "exp_v", "exp_r"])
v, r = classify(cases["adv"].astype(float), cases["map"].astype(float), cases["lpp"].astype(float), cases["promo"].astype(float))
assert list(v) == list(cases["exp_v"]), (list(v), list(cases["exp_v"]))
assert [x if x is not None else None for x in r] == list(cases["exp_r"]), list(r)
print("classify hand tests passed:", len(cases), "cases")

**What this cell does:** Applies the rules to every observation, then re-checks a random sample of 5,000 rows plus every unclassifiable row with a separate, deliberately simple row-by-row version of the same rules. Stops if even one row disagrees.

**Purpose:** The fast all-at-once version is what runs; the slow row-by-row version is the easy-to-read definition. Proving they agree on real data means the speed-up did not change any result.

In [ ]:
pm["is_violation"], pm["violation_reason"] = classify(pm["Advertised_price_base"], pm["MAP_Price"], pm["LPP"], pm["Promotional_price"])

def classify_row(adv, map_p, lpp, promo):            # the plain, slow reference version
    if pd.isna(adv) or pd.isna(map_p) or pd.isna(lpp):
        return False, R_UNCLASS
    if adv < lpp:
        return True, R_LPP
    if pd.notna(promo) and adv >= promo:
        return False, None
    if adv < map_p:
        return True, R_MAP
    return False, None

check = pd.concat([pm.sample(min(5000, len(pm)), random_state=1), pm[pm["violation_reason"] == R_UNCLASS]]).drop_duplicates()
mismatch = 0
for r_ in check.itertuples():
    exp = classify_row(r_.Advertised_price_base, r_.MAP_Price, r_.LPP, r_.Promotional_price)
    got_reason = r_.violation_reason if isinstance(r_.violation_reason, str) else None   # empty = None
    if (bool(r_.is_violation), got_reason) != exp:
        mismatch += 1
assert mismatch == 0, f"{mismatch} row(s) differ between fast and reference classification"
print(f"Self-check passed on {len(check):,} rows: fast and row-by-row rules agree on every row.")

**What this cell does:** Prints the headline numbers (rows judged, violations, split by hard-floor vs MAP breach, rows that cannot be judged) and breaks the unjudgeable rows down by cause.

**Purpose:** A quick sanity read of the result before it is stored, and an early warning if a data gap (missing price list entry, no LPP rule, unknown region) is hiding products from monitoring.

In [ ]:
total = len(pm); n_v = int(pm["is_violation"].sum())
n_un = int((pm["violation_reason"] == R_UNCLASS).sum())
print(f"Observations                : {total:,}")
print(f"Violations                  : {n_v:,} ({n_v / total:.1%})")
print(f"   below LPP (hard floor)   : {int((pm['violation_reason'] == R_LPP).sum()):,}")
print(f"   below MAP (no promo cover): {int((pm['violation_reason'] == R_MAP).sum()):,}")
print(f"Cannot be judged            : {n_un:,}")
un = pm[pm["violation_reason"] == R_UNCLASS].copy()
un["cause"] = np.select([un["MAP_Price"].isna(), un["LPP"].isna(), un["currency_local"].isna() | un["Advertised_price_base"].isna()],
                        ["no price list entry (no MAP)", "no LPP rule for this company/sub-category", "unknown region/currency or bad price"], "other")
print("\nCannot-be-judged rows by cause:"); print(un["cause"].value_counts().to_string())

**What this cell does:** Saves the list of products that could not be judged (SKU, product line, category, cause, number of observations) to `outputs/reports/unclassifiable_skus.csv`.

**Purpose:** These products are invisible to enforcement until the data gap is fixed. This file is the fix-list for the data owner (e.g. add a price list entry, or define an LPP percentage for scanners).

In [ ]:
if len(un):
    fix = (un.groupby(["SKU", "PL", "CATEGORY", "SUBCATEGORY", "cause"], dropna=False).size().rename("observations")
             .reset_index().sort_values("observations", ascending=False))
    path = os.path.join(mc.REPORT_DIR, "unclassifiable_skus.csv"); fix.to_csv(path, index=False)
    print(f"{len(fix)} SKU(s) -> {os.path.relpath(path, mc.PROJECT_DIR)}")
else:
    fix = pd.DataFrame(); print("Every observation could be judged.")
fix.head(10)

**What this cell does:** Arranges the result into the 21 columns of `PRICE_MONITORING_TABLE`, drops and re-creates that table, and loads all rows.

**Purpose:** Stores the verdict for every observation. The table is fully recalculated each run (it is derived data), so it always reflects the latest master data, promotions and rates.

In [ ]:
pm_final = pm.rename(columns={"file_date": "Violation_date", "region": "Region", "Seller_name": "seller_name", "CATEGORY": "Category",
                              "SUBCATEGORY": "Sub_category", "adv_price": "Advertised_price", "MAP_Price": "MAP_Price"})[[
    "SKU", "Violation_date", "PL", "Category", "Sub_category", "Region", "Marketplace", "seller_name", "homologated_name",
    "MAP_Price", "LPP", "Advertised_price", "currency_local", "Advertised_price_base", "Season", "Promotional_price",
    "promo_dollar_base", "fx_rate_used", "is_violation", "violation_reason"]].copy()
pm_final["Violation_date"] = pm_final["Violation_date"].dt.date

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS PRICE_MONITORING_TABLE"))
    conn.execute(text("""
        CREATE TABLE PRICE_MONITORING_TABLE (
            monitoring_id          INT AUTO_INCREMENT PRIMARY KEY,
            SKU                    VARCHAR(50) NOT NULL,
            Violation_date         DATE NOT NULL,
            PL                     VARCHAR(100),
            Category               VARCHAR(100),
            Sub_category           VARCHAR(100),
            Region                 VARCHAR(50),
            Marketplace            VARCHAR(150),
            seller_name            VARCHAR(255),
            homologated_name       VARCHAR(255),
            MAP_Price              DECIMAL(12,2),
            LPP                    DECIMAL(12,2),
            Advertised_price       DECIMAL(12,2),
            currency_local         VARCHAR(10),
            Advertised_price_base  DECIMAL(12,2),
            Season                 VARCHAR(10),
            Promotional_price      DECIMAL(12,2),
            promo_dollar_base      DECIMAL(12,2),
            fx_rate_used           DECIMAL(12,8),
            is_violation           BOOLEAN,
            violation_reason       VARCHAR(255),
            INDEX IX_PM_DATE (Violation_date),
            INDEX IX_PM_VIOL (is_violation),
            CONSTRAINT FK_PRICEMON_SKU FOREIGN KEY (SKU) REFERENCES SKU_TABLE(SKU)
        ) ENGINE=InnoDB"""))
    pm_final.to_sql("PRICE_MONITORING_TABLE", con=conn, if_exists="append", index=False, chunksize=1000, method="multi")
print(f"Loaded {len(pm_final):,} rows into PRICE_MONITORING_TABLE.")

**What this cell does:** Reads the table back and confirms the row count and violation count equal what was calculated, and shows violations per day.

**Purpose:** Final proof that what is stored is what was computed. Notebooks 06 and 07 read from this table.

In [ ]:
with engine.connect() as conn:
    n_db = conn.execute(text("SELECT COUNT(*) FROM PRICE_MONITORING_TABLE")).scalar()
    v_db = conn.execute(text("SELECT COALESCE(SUM(is_violation), 0) FROM PRICE_MONITORING_TABLE")).scalar()
    daily = pd.read_sql(text("""SELECT Violation_date AS day, COUNT(*) AS observations, SUM(is_violation) AS violations
                                FROM PRICE_MONITORING_TABLE GROUP BY Violation_date ORDER BY Violation_date"""), conn)
print(f"Rows in table: {n_db:,} (expected {len(pm_final):,}) | violations: {int(v_db):,} (expected {n_v:,})")
assert n_db == len(pm_final) and int(v_db) == n_v, "Stored results differ from calculated results"
daily["violation_rate_%"] = (daily["violations"].astype(float) / daily["observations"] * 100).round(1)
daily

**What this cell does:** Saves a backup of the database to Drive.

**Purpose:** So notebooks 06 and 07 (new Colab computers) start with the results already in place.

In [ ]:
mc.db_backup()

### Done
Next: **06_Enforcement_Letters_and_Summary** — turn violations into warnings and violation letters.